# Časť 3, Téma 1: Veľké výkyvy Hammingovej váhy (MAIN)

---
POZNÁMKA: Toto praktické cvičenie odkazuje na niektoré (komerčné) školiace materiály na webe [ChipWhisperer.io](https://www.ChipWhisperer.io). Cvičenie môžete voľne spustiť a používať v súlade s licenciou open-source (vrátane jeho použitia vo vašich vlastných kurzoch, ak ich budete distribuovať podobným spôsobom), musíte však zachovať informáciu o tomto pôvodnom zdroji. Ak si chcete vyskúšať plnohodnotný zážitok, zvážte účasť na našom školení.

---

**ZHRNUTIE:** *V predchádzajúcej časti kurzu ste videli, že spotreba energie mikrokontroléra sa mení v závislosti od toho, akú činnosť práve vykonáva. V prípade jednoduchej kontroly hesla nám to umožnilo zistiť, koľko znakov hesla bolo správnych, čo nakoniec viedlo k jeho prelomeniu.*

*Tento útok bol založený na tom, že rôzne vetvy vykonávania kódu sa v výkonových stopách (power traces) prejavovali odlišne. V nasledujúcej sérii cvičení budeme vychádzať z predpokladu, že spotrebu energie neovplyvňujú len samotné inštrukcie, ale aj dáta, s ktorými mikrokontrolér manipuluje.*


**VÝSLEDKY VZDELÁVANIA:**

* Využitie merania spotreby na „validáciu“ potenciálneho modelu zariadenia.
* Detekcia hodnoty jedného bitu pomocou merania spotreby.
* Prelomenie šifry AES pomocou klasického DPA útoku (Diferenciálna výkonová analýza).

## Prerekvizity

Moment! Skôr než budete pokračovať, skontrolujte, či máte úspešne absolvované tieto tútoriály:

* ☑ Úvod do Jupyter Notebookov (nemal by vám robiť problém beh kódových blokov a vykresľovanie grafov).
* ☑ Úvod do SCA101 (mali by ste mať prehľad o tom, ako spúšťať verzie špecifické pre daný hardvér).
* ☑ SCA101 Časť 2 (mali by ste rozumieť tomu, ako sa mení spotreba energie na základe spusteného kódu).

## Zber výkonových stôp

V tomto bode musíte vložiť kód na zachytávanie výkonových stôp. Máte dve možnosti:
* Zachytávať dáta z fyzického zariadenia.
* Čítať dáta zo súboru.


Uistite sa, že po spustení nasledujúcej bunky sa vám vypíše správa `"✔️ OK to continue!"`, inak neskoršie kroky zlyhajú!


In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEXMEGA'
CRYPTO_TARGET='AVRCRYPTOLIB'
SS_VER='SS_VER_2_1'

In [ ]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"
cd ../../firmware/mcu/simpleserial-aes
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

In [ ]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-aes/simpleserial-aes-{}.hex".format(PLATFORM))

In [ ]:
from tqdm.notebook import trange
import numpy as np
import time

ktp = cw.ktp.Basic()
trace_array = []
textin_array = []

key, text = ktp.next()

target.set_key(key)

N = 100
for i in trange(N, desc='Capturing traces'):
    scope.arm()
    if text[0] & 0x01:
        text[0] = 0xFF
    else:
        text[0] = 0x00
    target.simpleserial_write('p', text)
    
    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue
    
    response = target.simpleserial_read('r', 16)
    
    trace_array.append(scope.get_last_trace())
    textin_array.append(text)
    
    key, text = ktp.next() 

In [ ]:
assert len(trace_array) == 100
print("✔️ OK to continue!")

## Rozdeľovanie stôp do skupín

Ako sme si ukázali na prezentácii, vychádzame z predpokladu, že nastavovanie bitov na dátových zberniciach spotrebúva merateľné množstvo energie. Teraz túto teóriu otestujeme tak, že necháme náš cieľ manipulovať s dátami, ktoré majú veľmi vysokú Hammingovu váhu (0xFF) a veľmi nízku Hammingovu váhu (0x00). Za týmto účelom cieľové zariadenie momentálne vykonáva šifrovanie AES pre text, ktorý sme mu poslali. Ak je náš predpoklad správny, mali by sme vidieť merateľný rozdiel medzi výkonovými stopami s vysokou a nízkou Hammingovou váhou.

V tejto chvíli sú všetky tieto stopy pomiešané. Rozdeľte ich do dvoch skupín: `one_list` a `zero_list`. Tu je ukážka toho, ako pomocou prvého bajtu skontrolovať hodnotu 0x00 a ak to táto hodnota nie je, predpokladať, že ide o 0xFF. Tu je jednoduchá iterácia na ich výpis:

In [ ]:
for i in range(len(trace_array)):
    if textin_array[i][0] == 0x00:
        print("This should be added to 0 list")
    else:
        print("This should be added to 1 list")

Teraz to rozšírte tak, aby sa prvky pridávali do dvoch polí: `one_list` a `zero_list`:

In [ ]:
      
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")


assert len(one_list) > len(zero_list)/2
assert len(zero_list) > len(one_list)/2

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    one_list = []
    zero_list = []
    for i in range(len(trace_array)):
        if textin_array[i][0] == 0x00:
            zero_list.append(trace_array[i])
        else:
            one_list.append(trace_array[i])


    assert len(one_list) > len(zero_list)/2
    assert len(zero_list) > len(one_list)/2
</code></pre>
</details>

Mali by sme mať dva rôzne zoznamy. To, či sme poslali 0xFF alebo 0x00, bolo náhodné, takže tieto zoznamy pravdepodobne nebudú rovnomerne rozdelené. Následne budeme chcieť urobiť priemer každej skupiny (uistite sa, že robíte priemer pre každý bod stopy zvlášť! Nechceme celkový priemer hodnôt stôp v čase). To nám pomôže vyhladiť prípadné anomálie a zároveň vyrieši problém s odlišným počtom stôp v jednotlivých skupinách.

Najjednoduchší spôsob, ako to dosiahnuť, je použiť funkciu `np.mean()`, ktorá akceptuje zoznam ako argument. Budete musieť špecifikovať aj parameter `axis`, aby ste zabezpečili výpočet v správnej dimenzii. Skontrolujte výslednú veľkosť, aby ste sa uistili, že stopy majú rovnakú dĺžku ako na vstupe – nasledujúci blok ukazuje, ako to môžete overiť za predpokladu, že ste pre priemer použili premennú `one_avg`.

In [ ]:
trace_length = len(one_list[0])
print("Traces had original sample length of %d"%trace_length)

# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")


if len(one_avg) != trace_length:
    raise ValueError("Average length is only %d - check you did correct dimensions!"%one_avg)

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    
    trace_length = len(one_list[0])
    print("Traces had original sample length of %d"%trace_length)

    one_avg = np.mean(one_list, axis=0)


    if len(one_avg) != trace_length:
    raise ValueError("Average length is only %d - check you did correct dimensions!"%one_avg)
</code></pre>
</details>

Na záver odčítajte tieto dva priemery a vykreslite výsledné dáta do grafu:

In [ ]:
# ###################
# Add your code here
# ###################
raise NotImplementedError("Add Your Code Here")

<details>
<summary>Nejaké riešenie</summary>
<pre><code>
    
    %matplotlib widget
    import matplotlib.pyplot as plt
    import mplcursors
    import numpy as np

    plt.figure(figsize=(10,4))
    one_avg = np.mean(one_list, axis=0)
    zero_avg = np.mean(zero_list, axis=0)

    plt.plot(one_avg - zero_avg)
                

    #Enable hover labels
    #cursor = mplcursors.cursor(lines, hover=True)
    #cursor.connect("add", lambda sel: sel.annotation.set_text(sel.artist.get_label()))

    plt.show()
</code></pre>
</details>

Blízko začiatku grafu by ste mali vidieť veľmi výraznú špičku. To znamená, že dáta, s ktorými sa v cieľovom zariadení manipuluje, sú viditeľné v jeho výkonovej stope! Opäť je tu veľký priestor na experimentovanie:

* Skúste nastaviť viacero bajtov na hodnoty 0x00 a 0xFF.
* Skúste použiť menšie rozdiely v Hammingovej váhe. Je táto špička stále zreteľná? Čo ak nasnímate viac stôp?
* V tomto cvičení sme sa zamerali na prvý bajt. Skúste vykresliť grafy rozdielov pre viacero rôznych bajtov do jedného spoločného grafu.
* Cieľ tu spracováva šifru AES. Dokážete dosiahnuť, aby sa špičky objavili na iných miestach, ak nastavíte bajt na hodnotu 0x00 alebo 0xFF v neskoršom kole AES (napríklad v 5. kole)?

Ešte jedna poznámka, ktorá by vás mohla zmiasť: nezabúdajte, že napätie meriate na vstupe do prístroja ChipWhisperer, pričom sa meria na snímacom odpore (shunt rezistor). Konečným dôsledkom toho je, že nižšie napätie v skutočnosti znamená vyšší odber prúdu (výkon). V grafe preto môžete vidieť špičky otočené opačným smerom, než by ste logicky očakávali na základe toho, že logická 1 spotrebúva viac energie ako logická 0.

---
<small>UPOZORNENIE: Tento materiál je chránený autorskými právami (C) NewAE Technology Inc., 2015–2020. ChipWhisperer je ochranná známka spoločnosti NewAE Technology Inc., na ktorú si táto spoločnosť uplatňuje práva vo všetkých jurisdikciách a ktorá je registrovaná minimálne v Spojených štátoch amerických, Európskej únii a Čínskej ľudovej republike.

Návody odvodené z našej open-source práce musia byť uverejnené pod príslušnou open-source licenciou a upozornenie na zdroj musí byť *jasne zobrazené*. Iba pôvodní držitelia autorských práv môžu udeľovať licencie alebo povoľovať ďalšiu distribúciu – hoci spoločnosť NewAE Technology Inc. vlastní autorské práva k mnohým návodom, repozitár github obsahuje príspevky komunity, ktoré nemôže licencovať za špeciálnych podmienok a **musia** byť udržiavané ako open-source verzia. V prípade potreby prosím kontaktujte NewAE Technology Inc ohľadom špeciálnych povolení (ak je to možné)

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.</small>